In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType, TimestampType
from pyspark.sql import Row

catalog = "Ecommerce_BI"

## Applying normalization for better readability in gold layer

### Creating views to join tables

In [0]:
df_brands = spark.read.table(f"{catalog}.silver.slvr_brands")
df_categories = spark.read.table(f'{catalog}.silver.slvr_categories')
df_products = spark.read.table(f'{catalog}.silver.slvr_products')

In [0]:
df_brands.createOrReplaceTempView("v_brands")
df_categories.createOrReplaceTempView("v_categories")
df_products.createOrReplaceTempView("v_products")

In [0]:
%sql
CREATE OR REPLACE TABLE Ecommerce_BI.gold.gld_products AS

WITH brands_categories AS (
SELECT
    b.brand_name,
    b.brand_code,
    c.category_name,
    c.category_code
    FROM
    v_brands b
    INNER JOIN
    v_categories c
    ON b.category_code = c.category_code
)
SELECT
    p.product_id,
    p.sku,
    P.category_code,
    COALESCE(bc.category_name, 'Unavailable') as category_name,
    p.brand_code,
    COALESCE(bc.brand_name, 'Unavailable') as brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.file_name,
    p.ingest_timestamp
    FROM v_products p
    LEFT JOIN brands_categories bc
    ON p.brand_code = bc.brand_code;

### For better Analytics we classify customer states into regions

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
# Flatten country_state_map into a list of Rows
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10] 

# Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)

# Show mapping
df_region_mapping.show(truncate=False)

In [0]:
df_customers = spark.read.table(f'{catalog}.silver.slvr_customers')
df_customers = df_customers.join(df_region_mapping, on = ['country', 'state'], how = 'left')
df_customers = df_customers.fillna({'region' : 'Other'})
display(df_customers.limit(5))

In [0]:
df_customers.write.format('delta').mode('overwrite').option('mergeSchema', True).saveAsTable(f'{catalog}.gold.gld_customers')

###Date Table

In [0]:
df_date = spark.read.table(f'{catalog}.silver.slvr_date')
df_date.show(5)

In [0]:
df_gold = df_date.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast('int'))

df_gold = df_gold.withColumn("month_name", F.date_format(F.col('date'), "MMMM"))

df_gold = df_gold.withColumn("isWeekend", F.when(F.col("day_name").isin(["Saturday", "Sunday"]), 1).otherwise(0))

display(df_gold.limit(5))

In [0]:
column_order = ["date_id", "date", "year", "month_name", "day_name", "isWeekend", "quarter", "week",	"_source_file", "ingested_at"]

df_gold = df_gold.select(column_order)

display(df_gold)

In [0]:
df_gold.write.format('delta').mode('overwrite').option('mergeShema', True).saveAsTable(f'{catalog}.gold.gld_date')